# NLP Homework Challenge — Classification Model Comparison

Part 7 homework: run the other classification models from Part 3 on the restaurant reviews Bag-of-Words data, evaluate Accuracy/Precision/Recall/F1, and try a model not covered in Part 3 (a boosted-tree model, as an available stand-in for C5.0; note that Logistic Regression *is* the Maximum Entropy classifier, and scikit-learn's Decision Tree already implements CART).

## Importing the libraries

In [1]:
import re
import numpy as np
import pandas as pd
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

## Importing the dataset

In [2]:
dataset = pd.read_csv('Restaurant_Reviews.tsv', delimiter = '\t', quoting = 3)

## Cleaning the texts

Same cleaning as the tutorial: strip non-letters, lowercase, remove stopwords (keeping *not*), and stem with the Porter stemmer.

In [3]:
corpus = []
ps = PorterStemmer()
all_stopwords = stopwords.words('english')
all_stopwords.remove('not')
for i in range(0, 1000):
  review = re.sub('[^a-zA-Z]', ' ', dataset['Review'][i])
  review = review.lower()
  review = review.split()
  review = [ps.stem(word) for word in review if not word in set(all_stopwords)]
  review = ' '.join(review)
  corpus.append(review)

## Creating the Bag of Words model

Using the same features as the tutorial (`max_features = 1500`, `ngram_range = (1, 2)`) so every model is compared on identical input — only the classifier changes.

In [4]:
cv = CountVectorizer(max_features = 1500, ngram_range = (1, 2))
X = cv.fit_transform(corpus).toarray()
y = dataset.iloc[:, -1].values

## Splitting the dataset into the Training set and Test set

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20, random_state = 0)

## Training and evaluating every classification model

- **Naive Bayes** — the tutorial's baseline, included for reference.
- **Logistic Regression** — also known as the **Maximum Entropy** classifier.
- **K-Nearest Neighbors**
- **SVM (linear kernel)**
- **Kernel SVM (RBF)**
- **Decision Tree** — this *is* the **CART** algorithm.
- **Random Forest**
- **Gradient Boosting** — a boosted-tree model in the spirit of **C5.0** (C5.0 itself is an R-only implementation with no scikit-learn/Python equivalent, so gradient-boosted trees are used as the closest available stand-in).

All models use the same hyperparameters as their respective Part 3 tutorials.

In [6]:
models = {
    "Naive Bayes (tutorial baseline)": MultinomialNB(),
    "Logistic Regression / Max Entropy": LogisticRegression(random_state = 0, max_iter = 1000),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors = 5, metric = 'minkowski', p = 2),
    "SVM (linear kernel)": SVC(kernel = 'linear', random_state = 0),
    "Kernel SVM (rbf)": SVC(kernel = 'rbf', random_state = 0),
    "Decision Tree / CART": DecisionTreeClassifier(criterion = 'entropy', random_state = 0),
    "Random Forest": RandomForestClassifier(n_estimators = 10, criterion = 'entropy', random_state = 0),
    "Gradient Boosting (C5.0-style boosted trees)": GradientBoostingClassifier(random_state = 0),
}

rows = []
for name, clf in models.items():
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    rows.append([name, acc, prec, rec, f1, cm])

results = pd.DataFrame(rows, columns=["Model", "Accuracy", "Precision", "Recall", "F1 Score", "Confusion Matrix"])
results_sorted = results.sort_values("F1 Score", ascending=False).reset_index(drop=True)
results_sorted

,Model,Accuracy,Precision,Recall,F1 Score,Confusion Matrix
0,Naive Bayes (tutorial baseline),0.800,0.805825,0.805825,0.805825,"[[77, 20], [20, 83]]"
1,Logistic Regression / Max Entropy,0.790,0.827957,0.747573,0.785714,"[[81, 16], [26, 77]]"
2,Random Forest,0.765,0.818182,0.699029,0.753927,"[[81, 16], [31, 72]]"
3,SVM (linear kernel),0.755,0.787234,0.718447,0.751269,"[[77, 20], [29, 74]]"
4,Kernel SVM (rbf),0.755,0.864865,0.621359,0.723164,"[[87, 10], [39, 64]]"
5,Decision Tree / CART,0.700,0.726316,0.669903,0.696970,"[[71, 26], [34, 69]]"
6,Gradient Boosting (C5.0-style boosted trees),0.725,0.887097,0.533981,0.666667,"[[90, 7], [48, 55]]"
7,K-Nearest Neighbors,0.630,0.773585,0.398058,0.525641,"[[85, 12], [62, 41]]"


## Results

Ranked by F1 Score (the balance between Precision and Recall). None of the additional models beat the tutorial's Naive Bayes on this exact Bag-of-Words representation — Logistic Regression (the Maximum Entropy classifier) comes closest, trading a bit of Recall for noticeably higher Precision.

In [7]:
for _, r in results_sorted.iterrows():
    print(f"{r['Model']}")
    print(f"  Accuracy={r['Accuracy']:.4f}  Precision={r['Precision']:.4f}  Recall={r['Recall']:.4f}  F1={r['F1 Score']:.4f}")
    print(f"  Confusion Matrix: {r['Confusion Matrix'].tolist()}")
    print()

Naive Bayes (tutorial baseline)
  Accuracy=0.8000  Precision=0.8058  Recall=0.8058  F1=0.8058
  Confusion Matrix: [[77, 20], [20, 83]]

Logistic Regression / Max Entropy
  Accuracy=0.7900  Precision=0.8280  Recall=0.7476  F1=0.7857
  Confusion Matrix: [[81, 16], [26, 77]]

Random Forest
  Accuracy=0.7650  Precision=0.8182  Recall=0.6990  F1=0.7539
  Confusion Matrix: [[81, 16], [31, 72]]

SVM (linear kernel)
  Accuracy=0.7550  Precision=0.7872  Recall=0.7184  F1=0.7513
  Confusion Matrix: [[77, 20], [29, 74]]

Kernel SVM (rbf)
  Accuracy=0.7550  Precision=0.8649  Recall=0.6214  F1=0.7232
  Confusion Matrix: [[87, 10], [39, 64]]

Decision Tree / CART
  Accuracy=0.7000  Precision=0.7263  Recall=0.6699  F1=0.6970
  Confusion Matrix: [[71, 26], [34, 69]]

Gradient Boosting (C5.0-style boosted trees)
  Accuracy=0.7250  Precision=0.8871  Recall=0.5340  F1=0.6667
  Confusion Matrix: [[90, 7], [48, 55]]

K-Nearest Neighbors
  Accuracy=0.6300  Precision=0.7736  Recall=0.3981  F1=0.5256
  Confus

## Conclusion / Justification

On this dataset and feature representation, **Naive Bayes (MultinomialNB)** — the model used in the tutorial — remains the strongest overall performer (Accuracy 0.800, F1 0.806), and none of the other Part 3 models or the added Gradient Boosting model beat it once Precision *and* Recall are both taken into account.

**Why Naive Bayes wins here:** the Bag-of-Words matrix is high-dimensional (1500 sparse word/bigram-count features) but the training set is tiny (800 reviews). Naive Bayes' strong conditional-independence assumption acts as an implicit regularizer — it needs very little data to estimate per-word class probabilities, which is exactly the regime text classification on small corpora falls into. Models with more flexible decision boundaries (SVM, Random Forest, Gradient Boosting, KNN) have more parameters to fit and overfit faster on 800 sparse, high-dimensional examples — visible in their lower Recall despite often *higher* Precision (e.g. Gradient Boosting: Precision 0.887 but Recall only 0.534 — it's very cautious about predicting "positive," catching fewer true positives).

**Runner-up:** Logistic Regression / Maximum Entropy is the closest competitor (Accuracy 0.790, F1 0.786) and would be my second choice — as a linear, well-regularized model it shares Naive Bayes' resistance to overfitting on sparse text features, and its Precision (0.828) is actually higher than Naive Bayes', at a modest Recall cost. If the goal shifted toward minimizing false positives (e.g. only flagging a review as "positive" when very confident), Logistic Regression/Max Entropy would be the better pick.

**KNN performs worst** (Accuracy 0.630, F1 0.526) — distance metrics degrade badly in 1500-dimensional sparse spaces (the curse of dimensionality), so it is a poor fit for Bag-of-Words text data.